# Week 1 — Checkpoint 4: Feature Engineering
### Dataset: `house_sale.csv` (bina.az — real estate sale listings)

https://www.kaggle.com/datasets/sehriyarmemmedli/binaaz-sale-project

**Goal:** Create at least 2 new, meaningful columns from the existing data. I created 4 conceptual feature groups (8 new columns in total) — explaining below why each is useful for real-estate analysis.

I first restore the output of the previous checkpoints (null-handling + outlier capping) and build on top of it.

## 1. Restoring the previous steps (Checkpoint 2 + 3)

In [2]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', 70)
pd.set_option('display.width', 150)

df = pd.read_csv('house_sale.csv')

df_clean = df.copy()
df_clean = df_clean.drop(columns=["Binanın növü", "hour_y"])
df_clean["featured_flag"] = df_clean["featured"].notnull()
df_clean["vip_flag"] = df_clean["vip"].notnull()
df_clean["mortgage_flag"] = df_clean["mortgage"].notnull()
df_clean["bill_of_sale_flag"] = df_clean["bill_of_sale"].notnull() | df_clean["Çıxarış"].notnull()
df_clean = df_clean.drop(columns=["featured", "vip", "mortgage", "İpoteka", "bill_of_sale", "Çıxarış", "repair"])
df_clean["Təmir"] = df_clean["Təmir"].fillna("Unknown")

def seller_type(row):
    if pd.notnull(row["shop_name"]):
        return "Agency"
    elif pd.notnull(row["owner_name"]):
        return "Individual"
    return "Unknown"

df_clean["seller_type"] = df_clean.apply(seller_type, axis=1)
for col in ["shop_name", "shop_title", "owner_name", "owner_title"]:
    df_clean[col] = df_clean[col].fillna("N/A")
df_clean["products_label"] = df_clean["products_label"].fillna("Regular listing")
df_clean["description"] = df_clean["description"].fillna("")

def parse_area(val):
    if pd.isnull(val):
        return np.nan
    val = val.strip()
    if "sot" in val:
        return float(val.replace("sot", "").strip().replace(",", ".")) * 100
    return float(val.replace("m²", "").strip().replace(" ", ""))

df_clean["Sahə_numeric"] = df_clean["Sahə"].apply(parse_area)
df_clean["unit_price_calculated"] = (df_clean["price"] / df_clean["Sahə_numeric"]).round(2)
df_clean = df_clean.drop(columns=["unit_price"])

applicable_categories = ["Köhnə tikili", "Yeni tikili", "Ofis", "Həyət evi/Bağ evi"]
mask = df_clean["Kateqoriya"].isin(applicable_categories)
medians = df_clean.loc[mask].groupby("Kateqoriya")["Otaq sayı"].median()
for cat, med in medians.items():
    idx = df_clean[(df_clean["Kateqoriya"] == cat) & (df_clean["Otaq sayı"].isnull())].index
    df_clean.loc[idx, "Otaq sayı"] = med

df_clean = df_clean[df_clean["price"] >= 1000].copy()

def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

def cap_grouped(df, col, group_col="Kateqoriya", k=1.5):
    capped = df[col].copy()
    flag = pd.Series(False, index=df.index)
    for cat, group in df.groupby(group_col):
        low, high = iqr_bounds(group[col].dropna(), k)
        idx = group.index
        below = df.loc[idx, col] < low
        above = df.loc[idx, col] > high
        flag.loc[idx[below | above]] = True
        capped.loc[idx[below]] = low
        capped.loc[idx[above]] = high
    return capped, flag

df_clean["price_capped"], df_clean["is_outlier_price"] = cap_grouped(df_clean, "price")
df_clean["Sahə_numeric_capped"], df_clean["is_outlier_area"] = cap_grouped(df_clean, "Sahə_numeric")
df_clean["unit_price_capped"], df_clean["is_outlier_unit_price"] = cap_grouped(df_clean, "unit_price_calculated")

low_v, high_v = iqr_bounds(df_clean["views"])
df_clean["views_capped"] = df_clean["views"].clip(low_v, high_v)
df_clean["is_outlier_views"] = (df_clean["views"] < low_v) | (df_clean["views"] > high_v)

print("df_clean is ready:", df_clean.shape)


df_clean is ready: (100765, 56)


## 2. Feature 1 — Floor-position columns (`floor_current`, `floor_total`, `floor_ratio`, `is_top_floor`, `is_ground_floor`)

**Why it's needed:** The `Mərtəbə` (floor) column arrives as text in the format `"7 / 9"` — unusable directly in calculations. I split it into two numbers (current floor / total floors) and derive **new meaningful indicators** from it:
- `floor_ratio` — the apartment's relative position in the building (close to 0 = lower floors, close to 1 = upper floors). This is a well-known price factor on bina.az-style markets (very low and very high floors are often priced somewhat lower).
- `is_top_floor` / `is_ground_floor` — the top floor (risk of roof leaks) and the ground floor (security/light concerns) are classic factors that negatively affect price in the real-estate market; flagging these separately as booleans is useful for future analysis/modeling.

In [3]:
floor_split = df_clean["Mərtəbə"].str.split("/", expand=True)
df_clean["floor_current"] = pd.to_numeric(floor_split[0].str.strip(), errors="coerce")
df_clean["floor_total"] = pd.to_numeric(floor_split[1].str.strip(), errors="coerce")
df_clean["floor_ratio"] = (df_clean["floor_current"] / df_clean["floor_total"]).round(2)
df_clean["is_top_floor"] = df_clean["floor_current"] == df_clean["floor_total"]
df_clean["is_ground_floor"] = df_clean["floor_current"] == 1

print(df_clean["floor_ratio"].describe())
print()
print("Share of listings on the top floor:", round(df_clean["is_top_floor"].mean() * 100, 2), "%")
print("Share of listings on the ground floor:", round(df_clean["is_ground_floor"].mean() * 100, 2), "%")


count    75990.000000
mean         0.590225
std          0.267435
min          0.030000
25%          0.370000
50%          0.600000
75%          0.820000
max          1.000000
Name: floor_ratio, dtype: float64

Share of listings on the top floor: 7.12 %
Share of listings on the ground floor: 2.69 %


## 3. Feature 2 — `location_type` (location category)

**Why it's needed:** The `city` column is "bakı" for 99.6% of rows — on its own, it has almost no discriminating power. But the suffix of the `location` values (`r.` = district center, `m.` = micro-district, `q.` = settlement) indirectly indicates the property's **proximity to/type relative to the city center** — settlements are typically farther out and relatively cheaper, while district centers tend to be pricier areas. I extract this suffix into a new categorical column.

In [4]:
def loc_type(loc):
    if pd.isnull(loc):
        return "Unknown"
    match = re.search(r"\s(r|m|q)\.$", loc.strip())
    mapping = {"r": "District center", "m": "Micro-district", "q": "Settlement"}
    return mapping.get(match.group(1), "Other") if match else "Other"

df_clean["location_type"] = df_clean["location"].apply(loc_type)
df_clean["location_type"].value_counts(dropna=False)


,count
location_type,
Micro-district,54158
Settlement,32626
District center,13642
Other,339


In [5]:
print("Median price_capped by location_type:")
print(df_clean.groupby("location_type")["price_capped"].median().sort_values(ascending=False))


Median price_capped by location_type:
location_type
District center    250000.0
Micro-district     230000.0
Settlement         173000.0
Other               87000.0
Name: price_capped, dtype: float64


**Observation:** The `District center` category has a clearly distinct median price from the others — confirming that `location_type` genuinely correlates with price and is a useful signal.

## 4. Feature 3 — `price_per_room`

**Why it's needed:** The raw `price` column alone doesn't show whether a property is "good value" — you can't say a 5-room house at 400,000 AZN is "cheaper" than a 2-room apartment at 300,000 AZN, since their sizes differ. `price_per_room` brings these two listings into a **comparable format** and answers the buyer's question of "how much am I paying per room" — a practical indicator commonly used on platforms like bina.az.

In [6]:
df_clean["price_per_room"] = np.where(
    df_clean["Otaq sayı"] > 0,
    (df_clean["price_capped"] / df_clean["Otaq sayı"]).round(0),
    np.nan
)
df_clean["price_per_room"].describe()


,price_per_room
count,91713.000000
mean,82989.155463
std,37778.725617
min,533.000000
25%,57500.000000
50%,78333.000000
75%,105000.000000
max,877500.000000


## 5. Feature 4 — `price_index_vs_category` (price index relative to category)

**Why it's needed:** A listing priced at 300,000 AZN doesn't by itself tell you whether it's "expensive" or "cheap" — that depends on its category (apartment, house, land, etc.). `price_index_vs_category` divides the listing's price by its **own category's median price** to create a relative index: 1.0 = equal to the category average, above 1.0 = pricier than its category, below = cheaper. This is a directly usable indicator for finding "good deals" (listings with a low index).

In [7]:
category_median = df_clean.groupby("Kateqoriya")["price_capped"].transform("median")
df_clean["price_index_vs_category"] = (df_clean["price_capped"] / category_median).round(2)

df_clean["price_index_vs_category"].describe()


,price_index_vs_category
count,100765.000000
mean,1.187402
std,0.831891
min,0.010000
25%,0.690000
50%,1.000000
75%,1.430000
max,6.020000


In [8]:
print("Top 5 'best value' listings (lowest relative price within their own category):")
df_clean.nsmallest(5, "price_index_vs_category")[["Kateqoriya", "city", "price_capped", "price_index_vs_category"]]


Top 5 'best value' listings (lowest relative price within their own category):


,Kateqoriya,city,price_capped,price_index_vs_category
9131,Torpaq,bakı,1300.0,0.01
12720,Obyekt,bakı,3200.0,0.01
13944,Torpaq,bakı,2200.0,0.01
14296,Torpaq,bakı,1300.0,0.01
16437,Obyekt,bakı,3200.0,0.01


## 6. Final check

Listing the newly created columns and confirming the dataset's final size.

In [9]:
new_features = [
    "floor_current", "floor_total", "floor_ratio", "is_top_floor", "is_ground_floor",
    "location_type", "price_per_room", "price_index_vs_category"
]
print("Newly created columns:", new_features)
print("Total column count now:", df_clean.shape[1])
df_clean[new_features].head()


Newly created columns: ['floor_current', 'floor_total', 'floor_ratio', 'is_top_floor', 'is_ground_floor', 'location_type', 'price_per_room', 'price_index_vs_category']
Total column count now: 64


,floor_current,floor_total,floor_ratio,is_top_floor,is_ground_floor,location_type,price_per_room,price_index_vs_category
0,7.0,9.0,0.78,False,False,District center,70625.0,1.95
1,NaN,NaN,NaN,False,False,Settlement,19250.0,0.33
2,NaN,NaN,NaN,False,False,Micro-district,30667.0,0.39
3,NaN,NaN,NaN,False,False,Settlement,NaN,0.16
4,15.0,16.0,0.94,False,False,Micro-district,73333.0,0.90


## Checkpoint 4 — Summary

Instead of the required minimum of 2 columns, I created 4 conceptual feature groups (8 new columns total), each answering a different analytical question:

1. **`floor_ratio`, `is_top_floor`, `is_ground_floor`** — converted the text-formatted `Mərtəbə` field into usable numeric/boolean indicators; allows measuring the effect of in-building position on price.
2. **`location_type`** — where `city` provided little value (99.6% identical), used the structure of `location` itself to distinguish location type (district center / micro-district / settlement); confirmed a clear relationship with median price.
3. **`price_per_room`** — brought properties of different sizes into a comparable format.
4. **`price_index_vs_category`** — created an index showing how "expensive/cheap" each listing is relative to its own category, directly usable for spotting good deals.
